In [7]:
!pip install pandas numpy tqdm python-docx

In [27]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter
import pprint

CANDIDATE_FILE = "/content/candidates.jsonl"
SCHEMA_FILE = "/content/candidate_schema.json"

print("="*80)
print("LOADING SCHEMA")
print("="*80)

try:
    with open(SCHEMA_FILE, "r") as f:
        schema = json.load(f)

    print("\nSCHEMA:")
    pprint.pprint(schema)

except Exception as e:
    print("Schema not loaded:", e)


print("\n" + "="*80)
print("COUNTING CANDIDATES")
print("="*80)

count = 0

with open(CANDIDATE_FILE, "r") as f:
    for _ in f:
        count += 1

print(f"\nTotal Candidates: {count:,}")

print("\n" + "="*80)
print("FIRST CANDIDATE")
print("="*80)

with open(CANDIDATE_FILE, "r") as f:
    first_candidate = json.loads(next(f))

print("\nTYPE:", type(first_candidate))

if isinstance(first_candidate, dict):
    print("\nTOP LEVEL KEYS:")
    print(list(first_candidate.keys()))

print("\n" + "="*80)
print("FIELD INSPECTION")
print("="*80)

for key, value in first_candidate.items():

    print("\n")
    print("-"*60)
    print("FIELD:", key)
    print("TYPE :", type(value))

    if isinstance(value, dict):
        print("SUBKEYS:")
        print(list(value.keys())[:20])

    elif isinstance(value, list):
        print("LIST LENGTH:", len(value))

        if len(value) > 0:
            print("FIRST ITEM:")
            pprint.pprint(value[0])

    else:
        print(value)

print("\n" + "="*80)
print("LOADING SAMPLE CANDIDATES")
print("="*80)

sample_size = min(5000, count)

sample_candidates = []

with open(CANDIDATE_FILE, "r") as f:

    for i, line in enumerate(f):

        if i >= sample_size:
            break

        try:
            sample_candidates.append(json.loads(line))
        except:
            pass

print(f"\nLoaded {len(sample_candidates):,} candidates")

print("\n" + "="*80)
print("TOP LEVEL FIELD FREQUENCY")
print("="*80)

field_counter = Counter()

for candidate in sample_candidates:

    if isinstance(candidate, dict):

        for key in candidate.keys():
            field_counter[key] += 1

field_df = pd.DataFrame(
    field_counter.items(),
    columns=["Field","Count"]
)

display(field_df.sort_values(
    "Count",
    ascending=False
))

print("\n" + "="*80)
print("TEXT FIELD DETECTION")
print("="*80)

def extract_text(obj):

    texts = []

    if isinstance(obj, str):
        texts.append(obj)

    elif isinstance(obj, list):

        for item in obj:
            texts.extend(extract_text(item))

    elif isinstance(obj, dict):

        for v in obj.values():
            texts.extend(extract_text(v))

    return texts

sample_texts = extract_text(first_candidate)

print(f"\nText snippets found: {len(sample_texts)}")

for i, txt in enumerate(sample_texts[:20]):
    print(f"\n[{i+1}]")
    print(txt[:200])

print("\n" + "="*80)
print("BUILDING PROFILE TEXT")
print("="*80)

def build_candidate_text(candidate):

    texts = extract_text(candidate)

    texts = [
        str(x)
        for x in texts
        if len(str(x).strip()) > 1
    ]

    return " ".join(texts)

sample_profile_text = build_candidate_text(
    first_candidate
)

print(sample_profile_text[:3000])
print("\n" + "="*80)
print("AI KEYWORD ANALYSIS")
print("="*80)

keywords = [
    "llm",
    "machine learning",
    "deep learning",
    "retrieval",
    "ranking",
    "embedding",
    "search",
    "recommendation",
    "python",
    "tensorflow",
    "pytorch",
    "transformer",
    "vector",
    "nlp"
]

keyword_counts = {}

for keyword in keywords:

    hits = 0

    for candidate in sample_candidates:

        text = build_candidate_text(
            candidate
        ).lower()

        if keyword in text:
            hits += 1

    keyword_counts[keyword] = hits

keyword_df = pd.DataFrame(
    keyword_counts.items(),
    columns=["Keyword","Candidates"]
)

display(
    keyword_df.sort_values(
        "Candidates",
        ascending=False
    )
)

print("\n" + "="*80)
print("PROFILE LENGTH ANALYSIS")
print("="*80)

lengths = []

for candidate in sample_candidates:

    text = build_candidate_text(candidate)

    lengths.append(len(text))

lengths = pd.Series(lengths)

print(lengths.describe())

field_df.to_csv(
    "field_analysis.csv",
    index=False
)

keyword_df.to_csv(
    "keyword_analysis.csv",
    index=False
)

print("\n" + "="*80)
print("EDA COMPLETE")
print("="*80)

print("\nGenerated files:")
print("1. field_analysis.csv")
print("2. keyword_analysis.csv")

print("\nNEXT STEP:")
print("Paste the output showing:")
print("- Top level keys")
print("- Field inspection")
print("- Schema")

LOADING SCHEMA

SCHEMA:
{'$schema': 'http://json-schema.org/draft-07/schema#',
 'description': 'Schema for a single candidate profile in the Intelligent '
                'Candidate Discovery & Ranking Challenge dataset.',
 'properties': {'candidate_id': {'description': 'Unique identifier for the '
                                                'candidate. Format: '
                                                'CAND_XXXXXXX (7 digits).',
                                 'pattern': '^CAND_[0-9]{7}$',
                                 'type': 'string'},
                'career_history': {'items': {'properties': {'company': {'type': 'string'},
                                                            'company_size': {'enum': ['1-10',
                                                                                      '11-50',
                                                                                      '51-200',
                                                               

,Field,Count
0,candidate_id,5000
1,profile,5000
2,career_history,5000
3,education,5000
4,skills,5000
5,certifications,5000
6,languages,5000
7,redrob_signals,5000



TEXT FIELD DETECTION

Text snippets found: 69

[1]
CAND_0000001

[2]
Ira Vora

[3]
Backend Engineer | SQL, Spark, Cloud

[4]
Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home te

[5]
Toronto

[6]
Canada

[7]
Backend Engineer

[8]
Mindtree

[9]
10001+

[10]
IT Services

[11]
Mindtree

[12]
Backend Engineer

[13]
2024-03-08

[14]
IT Services

[15]
10001+

[16]
Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, 

[17]
Dunder Mifflin

[18]
Analytics Engineer

[19]
2019-07-03

[20]
2024-01-08

BUILDING PROFILE TEXT
CAND_0000001 Ira Vora Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructur

,Keyword,Candidates
0,llm,2087
6,search,1450
1,machine learning,686
8,python,658
12,vector,393
11,transformer,385
7,recommendation,280
3,retrieval,278
5,embedding,243
13,nlp,111



PROFILE LENGTH ANALYSIS
count    5000.000000
mean     2385.788200
std       691.080131
min      1209.000000
25%      1838.000000
50%      2300.000000
75%      2820.000000
max      5195.000000
dtype: float64

EDA COMPLETE

Generated files:
1. field_analysis.csv
2. keyword_analysis.csv

NEXT STEP:
Paste the output showing:
- Top level keys
- Field inspection
- Schema


In [28]:
# FEATURE ENGINEERING

!pip -q install pandas pyarrow tqdm

import json
import pandas as pd
import numpy as np
from tqdm import tqdm

INPUT_FILE = "/content/candidates.jsonl"

tier_map = {
    "tier_1": 4,
    "tier_2": 3,
    "tier_3": 2,
    "tier_4": 1,
    "unknown": 0
}

company_size_map = {
    "1-10": 1,
    "11-50": 2,
    "51-200": 3,
    "201-500": 4,
    "501-1000": 5,
    "1001-5000": 6,
    "5001-10000": 7,
    "10001+": 8
}

def build_candidate_text(candidate):

    profile = candidate.get("profile", {})
    career = candidate.get("career_history", [])
    skills = candidate.get("skills", [])
    education = candidate.get("education", [])

    parts = []

    parts.append(profile.get("headline", ""))
    parts.append(profile.get("summary", ""))

    for job in career:
        parts.append(job.get("title", ""))
        parts.append(job.get("description", ""))

    for skill in skills:
        parts.append(skill.get("name", ""))

    for edu in education:
        parts.append(edu.get("degree", ""))
        parts.append(edu.get("field_of_study", ""))

    return " ".join(parts)


rows = []

with open(INPUT_FILE, "r") as f:

    for line in tqdm(f):

        candidate = json.loads(line)

        profile = candidate.get("profile", {})
        skills = candidate.get("skills", [])
        career = candidate.get("career_history", [])
        education = candidate.get("education", [])
        signals = candidate.get("redrob_signals", {})

        # Experience

        years_exp = profile.get(
            "years_of_experience", 0
        )

        total_job_months = sum(
            job.get("duration_months", 0)
            for job in career
        )

        current_company_size = company_size_map.get(
            profile.get(
                "current_company_size",
                "1-10"
            ),
            0
        )

        # Education
        edu_tier = 0

        if len(education) > 0:

            edu_tier = max([
                tier_map.get(
                    edu.get("tier", "unknown"),
                    0
                )
                for edu in education
            ])

        # Skills

        skill_names = [
            s.get("name", "").lower()
            for s in skills
        ]

        skill_count = len(skill_names)

        llm_skill = int(
            any(
                "llm" in s
                for s in skill_names
            )
        )

        retrieval_skill = int(
            any(
                "retrieval" in s
                for s in skill_names
            )
        )

        ranking_skill = int(
            any(
                "ranking" in s
                for s in skill_names
            )
        )

        embedding_skill = int(
            any(
                "embedding" in s
                for s in skill_names
            )
        )

        vector_skill = int(
            any(
                "vector" in s
                for s in skill_names
            )
        )

        nlp_skill = int(
            any(
                "nlp" in s
                for s in skill_names
            )
        )

        # Behavioral Signals

        github_score = signals.get(
            "github_activity_score",
            -1
        )

        recruiter_response = signals.get(
            "recruiter_response_rate",
            0
        )

        interview_rate = signals.get(
            "interview_completion_rate",
            0
        )

        offer_rate = signals.get(
            "offer_acceptance_rate",
            0
        )

        profile_score = signals.get(
            "profile_completeness_score",
            0
        )

        search_count = signals.get(
            "search_appearance_30d",
            0
        )

        saved_count = signals.get(
            "saved_by_recruiters_30d",
            0
        )

        notice_period = signals.get(
            "notice_period_days",
            180
        )

        open_to_work = int(
            signals.get(
                "open_to_work_flag",
                False
            )
        )

        # Candidate Text

        candidate_text = build_candidate_text(
            candidate
        )

        # Row

        rows.append({

            "candidate_id":
                candidate["candidate_id"],

            "candidate_text":
                candidate_text,

            "years_exp":
                years_exp,

            "total_job_months":
                total_job_months,

            "company_size":
                current_company_size,

            "edu_tier":
                edu_tier,

            "skill_count":
                skill_count,

            "llm_skill":
                llm_skill,

            "retrieval_skill":
                retrieval_skill,

            "ranking_skill":
                ranking_skill,

            "embedding_skill":
                embedding_skill,

            "vector_skill":
                vector_skill,

            "nlp_skill":
                nlp_skill,

            "github_score":
                github_score,

            "recruiter_response":
                recruiter_response,

            "interview_rate":
                interview_rate,

            "offer_rate":
                offer_rate,

            "profile_score":
                profile_score,

            "search_count":
                search_count,

            "saved_count":
                saved_count,

            "notice_period":
                notice_period,

            "open_to_work":
                open_to_work

        })

# DataFrame

df = pd.DataFrame(rows)

print("\nShape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nPreview:")
display(df.head())

df.to_parquet(
    "candidate_features.parquet",
    index=False
)

df.to_csv(
    "candidate_features.csv",
    index=False
)

print("\nSaved:")
print("candidate_features.parquet")
print("candidate_features.csv")

print("\nDone.")

100000it [00:08, 12467.35it/s]



Shape: (100000, 22)

Columns:
['candidate_id', 'candidate_text', 'years_exp', 'total_job_months', 'company_size', 'edu_tier', 'skill_count', 'llm_skill', 'retrieval_skill', 'ranking_skill', 'embedding_skill', 'vector_skill', 'nlp_skill', 'github_score', 'recruiter_response', 'interview_rate', 'offer_rate', 'profile_score', 'search_count', 'saved_count', 'notice_period', 'open_to_work']

Preview:


,candidate_id,candidate_text,years_exp,total_job_months,company_size,edu_tier,skill_count,llm_skill,retrieval_skill,ranking_skill,...,nlp_skill,github_score,recruiter_response,interview_rate,offer_rate,profile_score,search_count,saved_count,notice_period,open_to_work
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud Software ...",6.9,82,8,2,17,1,0,0,...,1,9.2,0.34,0.71,0.58,86.9,249,4,60,1
1,CAND_0000002,Operations Manager | 12.5+ yrs experience Prof...,12.5,149,8,1,9,0,0,0,...,0,-1.0,0.29,0.62,-1.00,78.7,107,10,60,1
2,CAND_0000003,Customer Support | 1.1+ yrs experience Profess...,1.1,13,8,2,6,0,0,0,...,0,-1.0,0.46,0.86,-1.00,31.9,28,4,150,0
3,CAND_0000004,Marketing Manager | Driving business outcomes ...,3.8,44,4,2,10,0,0,0,...,0,-1.0,0.26,0.35,-1.00,28.5,5,8,120,0
4,CAND_0000005,Accountant | Helping teams scale Professional ...,11.0,130,6,2,6,0,0,0,...,0,-1.0,0.37,0.74,-1.00,84.6,67,1,30,1



Saved:
candidate_features.parquet
candidate_features.csv

Done.


In [31]:
# JD EMBEDDINGS + FAISS RETRIEVAL

!pip -q install sentence-transformers faiss-cpu

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

df = pd.read_parquet(
    "candidate_features.parquet"
)

print(df.shape)


model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


from docx import Document

doc = Document("/content/job_description.docx")

jd_text = "\n".join(
    [p.text for p in doc.paragraphs]
)

print(jd_text[:2000])

jd_query = jd_text


texts = df["candidate_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.array(
    embeddings,
    dtype=np.float32
)

print(
    "Embeddings:",
    embeddings.shape
)


np.save(
    "candidate_embeddings.npy",
    embeddings
)


dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)

faiss.write_index(
    index,
    "candidate_faiss.index"
)


jd_embedding = model.encode(
    [jd_query],
    normalize_embeddings=True
)

jd_embedding = np.array(
    jd_embedding,
    dtype=np.float32
)


TOP_K = 2000

scores, indices = index.search(
    jd_embedding,
    TOP_K
)

candidate_subset = df.iloc[
    indices[0]
].copy()

candidate_subset["semantic_score"] = (
    scores[0]
)

candidate_subset = (
    candidate_subset
    .sort_values(
        "semantic_score",
        ascending=False
    )
)

print(
    candidate_subset[
        [
            "candidate_id",
            "semantic_score",
            "years_exp"
        ]
    ].head(20)
)


candidate_subset.to_csv(
    "top2000_semantic.csv",
    index=False
)

print(
    "\nSaved top2000_semantic.csv"
)

print(
    "\nTop Candidate Text:"
)

print(
    candidate_subset.iloc[0]
    ["candidate_text"][:3000]
)

(100000, 22)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.
If you've spent your career bouncing between early-stage startups and you want to "just code" without having to think 

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Embeddings: (100000, 384)
       candidate_id  semantic_score  years_exp
94758  CAND_0094759        0.856871        8.6
2024   CAND_0002025        0.850050        5.9
39753  CAND_0039754        0.845635       16.2
93546  CAND_0093547        0.843069        2.9
46063  CAND_0046064        0.842830        8.9
5537   CAND_0005538        0.841797        5.9
33860  CAND_0033861        0.837595        8.0
46524  CAND_0046525        0.836902        6.1
86021  CAND_0086022        0.834128        5.3
41610  CAND_0041611        0.833177        6.4
92277  CAND_0092278        0.832234        6.8
71973  CAND_0071974        0.831465        7.8
18498  CAND_0018499        0.831318        7.2
77336  CAND_0077337        0.829003        7.0
55904  CAND_0055905        0.827871        8.1
81845  CAND_0081846        0.826820        6.7
8424   CAND_0008425        0.821254        7.8
4401   CAND_0004402        0.820010        6.0
74324  CAND_0074325        0.818876        3.9
21826  CAND_0021827        0.81778

In [32]:
# HYBRID RANKING ENGINE
import pandas as pd
import numpy as np


df = pd.read_csv("top2000_semantic.csv")

print("Input Shape:", df.shape)

# NORMALIZATION

def normalize(series):

    mn = series.min()
    mx = series.max()

    if mx == mn:
        return pd.Series(
            np.zeros(len(series)),
            index=series.index
        )

    return (series - mn) / (mx - mn)

# EXPERIENCE SCORE

def experience_score(years):

    if 5 <= years <= 9:
        return 1.0

    elif 4 <= years < 5:
        return 0.8

    elif 9 < years <= 12:
        return 0.7

    elif 3 <= years < 4:
        return 0.5

    elif 12 < years <= 15:
        return 0.4

    else:
        return 0.2

df["experience_score"] = (
    df["years_exp"]
    .apply(experience_score)
)

# BEHAVIOR SCORE

github = normalize(
    df["github_score"].clip(lower=0)
)

response = df["recruiter_response"]

interview = df["interview_rate"]

offer = (
    df["offer_rate"]
    .replace(-1, 0)
)

profile = normalize(
    df["profile_score"]
)

search = normalize(
    df["search_count"]
)

saved = normalize(
    df["saved_count"]
)

df["behavior_score"] = (

      0.20 * github
    + 0.20 * response
    + 0.15 * interview
    + 0.15 * offer
    + 0.10 * profile
    + 0.10 * search
    + 0.10 * saved

)

# AVAILABILITY SCORE

notice = df["notice_period"]

availability = np.where(
    notice <= 30,
    1.0,
    np.where(
        notice <= 60,
        0.8,
        np.where(
            notice <= 90,
            0.5,
            0.2
        )
    )
)

availability = (
    availability
    * (0.5 + 0.5 * df["open_to_work"])
)

df["availability_score"] = availability

# JD INTENT SCORE

bonus_terms = {

    # Retrieval
    "retrieval": 4,
    "dense retrieval": 6,
    "hybrid retrieval": 8,

    # Ranking
    "ranking": 4,
    "ranker": 5,
    "learning-to-rank": 8,

    # Recommendation
    "recommendation": 5,
    "recommendation system": 6,

    # Search
    "search": 3,
    "semantic search": 5,

    # Infra
    "faiss": 6,
    "pinecone": 6,
    "weaviate": 6,
    "qdrant": 6,
    "milvus": 6,
    "opensearch": 5,
    "elasticsearch": 5,

    # Embeddings
    "embedding": 4,
    "sentence transformers": 5,
    "bge": 5,
    "e5": 5,

    # Evaluation
    "ndcg": 8,
    "mrr": 8,
    "map": 6,
    "evaluation": 4,
    "offline benchmark": 6,
    "a/b": 5,
    "ab test": 5,

    # Product signals
    "marketplace": 4,
    "consumer product": 4,
    "e-commerce": 4,

    # Production
    "production ml": 6,
    "production system": 5,
    "production deployment": 5,

    # Behavioral ranking
    "behavioral signal": 6,
    "candidate matching": 5,

    # Desired
    "recruiter": 4,
    "matching": 4
}

penalty_terms = {

    # Explicit JD negatives

    "marketing manager": -15,
    "customer support": -15,
    "accountant": -15,
    "operations manager": -12,
    "sales executive": -12,
    "graphic designer": -12,

    # Research heavy

    "research assistant": -10,
    "academic researcher": -10,
    "phd researcher": -8,

    # Wrong domains

    "robotics": -5,
    "speech recognition": -5,

    # Consulting-only indicators

    "tcs": -3,
    "infosys": -3,
    "wipro": -3,
    "cognizant": -3,
    "capgemini": -3,

    # JD specifically calls this out

    "langchain tutorial": -10
}

intent_scores = []

for text in df["candidate_text"]:

    text = str(text).lower()

    score = 0

    for term, weight in bonus_terms.items():

        if term in text:
            score += weight

    for term, weight in penalty_terms.items():

        if term in text:
            score += weight

    intent_scores.append(score)

df["intent_score_raw"] = intent_scores

df["intent_score"] = normalize(
    df["intent_score_raw"]
)

# SKILL BONUS

skill_bonus = (

      2 * df["retrieval_skill"]
    + 2 * df["ranking_skill"]
    + 2 * df["embedding_skill"]
    + 1 * df["vector_skill"]
    + 1 * df["llm_skill"]
    + 1 * df["nlp_skill"]

)

df["skill_bonus"] = normalize(
    skill_bonus
)

# HONEYPOT / SUSPICIOUS SCORE

suspicious = np.zeros(len(df))

# Too much experience for role

suspicious += np.where(
    df["years_exp"] > 15,
    1,
    0
)

# Very low profile quality

suspicious += np.where(
    df["profile_score"] < 20,
    1,
    0
)

# Very low recruiter engagement

suspicious += np.where(
    df["recruiter_response"] < 0.05,
    1,
    0
)

# Not open to work

suspicious += np.where(
    df["open_to_work"] == 0,
    1,
    0
)

df["suspicious_score"] = suspicious

# SEMANTIC SCORE

df["semantic_score_norm"] = normalize(
    df["semantic_score"]
)

# FINAL SCORE

df["final_score"] = (

      0.38 * df["semantic_score_norm"]

    + 0.20 * df["behavior_score"]

    + 0.15 * df["experience_score"]

    + 0.10 * df["availability_score"]

    + 0.12 * df["intent_score"]

    + 0.05 * df["skill_bonus"]

)

# Honeypot penalty

df["final_score"] = (
    df["final_score"]
    - 0.03 * df["suspicious_score"]
)

# SORT

df = df.sort_values(
    "final_score",
    ascending=False
)

# TOP 100

top100 = df.head(100).copy()

# DISPLAY

print("\nTOP 20 CANDIDATES\n")

display(
    top100[
        [
            "candidate_id",
            "final_score",
            "semantic_score",
            "intent_score",
            "behavior_score",
            "experience_score",
            "years_exp",
            "notice_period"
        ]
    ].head(20)
)

# SAVE RANKED OUTPUT

top100.to_csv(
    "top100_ranked_final.csv",
    index=False
)

# CREATE SUBMISSION

submission = pd.DataFrame({
    "candidate_id":
        top100["candidate_id"],
    "rank":
        np.arange(
            1,
            len(top100) + 1
        )
})

submission.to_csv(
    "submission_final.csv",
    index=False
)

print("\nSaved:")
print("top100_ranked_final.csv")
print("submission_final.csv")

# TOP CANDIDATE INSPECTION

print("\n")
print("="*100)
print("TOP CANDIDATE")
print("="*100)

print(
    top100.iloc[0]["candidate_text"][:3000]
)

Input Shape: (2000, 23)

TOP 20 CANDIDATES



,candidate_id,final_score,semantic_score,intent_score,behavior_score,experience_score,years_exp,notice_period
1,CAND_0002025,0.856902,0.850050,0.845411,0.717627,1.0,5.9,30
12,CAND_0018499,0.799436,0.831318,0.961353,0.754404,1.0,7.2,15
8,CAND_0086022,0.798231,0.834128,0.961353,0.635148,1.0,5.3,0
4,CAND_0046064,0.787877,0.842830,0.966184,0.567228,1.0,8.9,30
0,CAND_0094759,0.781142,0.856871,0.903382,0.488680,1.0,8.6,30
7,CAND_0046525,0.773028,0.836902,0.879227,0.587780,1.0,6.1,60
13,CAND_0077337,0.743268,0.829003,0.908213,0.664399,1.0,7.0,60
15,CAND_0081846,0.743194,0.826820,0.951691,0.551864,1.0,6.7,30
11,CAND_0071974,0.742142,0.831465,0.893720,0.604778,1.0,7.8,45
5,CAND_0005538,0.742008,0.841797,0.608696,0.703683,1.0,5.9,90



Saved:
top100_ranked_final.csv
submission_final.csv


TOP CANDIDATE
Senior AI Engineer | Building AI-native search & ranking systems Senior AI engineer with 5.9 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector recall, across a corpus of 30M+ candidate profiles. My day-to-day work spans embedding model selection and fine-tuning, hybrid retrieval architecture, learning-to-rank, behavioral-signal integration, and the offline/online evaluation that ties it all together. I've shipped systems in both early-stage product companies and at larger scale, and I've spent enough time on both that I know which tradeoffs apply where. I care more about shipping a working system in 6 weeks than a theoretically perfect one in 6 months. Currently exploring my next move — looking for senior IC or tech-lead roles where I can own the intelligence

In [33]:
import pandas as pd
import numpy as np

top100 = pd.read_csv("top100_ranked_final.csv")


scores = top100["final_score"]

submission_score = (
    (scores - scores.min())
    /
    (scores.max() - scores.min())
)

submission_score = 0.60 + submission_score * 0.39

top100["submission_score"] = submission_score


top100 = top100.sort_values(
    ["submission_score", "candidate_id"],
    ascending=[False, True]
).reset_index(drop=True)


def generate_reasoning(row):

    reasons = []

    if row["intent_score"] > 0.8:
        reasons.append("strong retrieval/ranking experience")

    if row["behavior_score"] > 0.65:
        reasons.append("high recruiter engagement")

    if row["notice_period"] <= 30:
        reasons.append("short notice period")

    if row["semantic_score"] > 0.85:
        reasons.append("excellent JD alignment")

    if not reasons:
        reasons.append("good overall fit")

    return (
        f"{row['years_exp']:.1f} yrs experience; "
        + "; ".join(reasons)
    )


submission = pd.DataFrame({
    "candidate_id": top100["candidate_id"],
    "rank": np.arange(1, len(top100)+1),
    "score": np.round(
        top100["submission_score"],
        8
    ),
    "reasoning": top100.apply(
        generate_reasoning,
        axis=1
    )
})


submission["score"] = (
    submission["score"]
    - np.arange(len(submission)) * 1e-8
)


submission.to_csv(
    "submission_final.csv",
    index=False
)

print(submission.head())
print("\nShape:", submission.shape)

   candidate_id  rank     score  \
0  CAND_0002025     1  0.990000   
1  CAND_0018499     2  0.929082   
2  CAND_0086022     3  0.927804   
3  CAND_0046064     4  0.916829   
4  CAND_0094759     5  0.909689   

                                           reasoning  
0  5.9 yrs experience; strong retrieval/ranking e...  
1  7.2 yrs experience; strong retrieval/ranking e...  
2  5.3 yrs experience; strong retrieval/ranking e...  
3  8.9 yrs experience; strong retrieval/ranking e...  
4  8.6 yrs experience; strong retrieval/ranking e...  

Shape: (100, 4)


In [34]:
!python validate_submission.py submission_final.csv

Submission is valid.
